# Chunking strategy (#22)

DistilRoBERTa reads at most 512 tokens at a time, and almost every CUAD contract is
longer than that. This notebook compares ways of cutting contracts into chunks and
picks one, measured by how many annotated clauses survive in one piece.

Everything uses the **train** split only. Test stays untouched.

In [1]:
# Loading the data
import json
import re
from bisect import bisect_left

import pandas as pd

with open("../data/cuad/train_separate_questions.json") as f:
    train = json.load(f)

# One entry per contract: its text and the (start, end) of every annotated span.
contracts = []
for doc in train["data"]:
    text = doc["paragraphs"][0]["context"]
    spans = []
    for qa in doc["paragraphs"][0]["qas"]:
        for a in qa["answers"]:
            spans.append((a["answer_start"], a["answer_start"] + len(a["text"])))
    contracts.append({"title": doc["title"], "text": text, "spans": spans})

total_spans = sum(len(c["spans"]) for c in contracts)
print("contracts:", len(contracts))
print("spans:", total_spans)

# Every span offset should point at exactly its answer text.
misaligned = 0
for c, doc in zip(contracts, train["data"]):
    for qa in doc["paragraphs"][0]["qas"]:
        for a in qa["answers"]:
            s = a["answer_start"]
            if c["text"][s:s + len(a["text"])] != a["text"]:
                misaligned += 1
print("misaligned spans:", misaligned)

contracts: 408
spans: 11180
misaligned spans: 0


In [2]:
# 1. How long are contracts, in the model's own tokens?
from transformers import AutoTokenizer, logging
logging.set_verbosity_error()      # silences "sequence longer than 512" warnings

tokenizer = AutoTokenizer.from_pretrained("distilroberta-base")

# 512 minus the 2 special tokens (<s> and </s>) the model adds to every input.
MAX_TOKENS = tokenizer.model_max_length - tokenizer.num_special_tokens_to_add()
print("tokens of contract text per chunk:", MAX_TOKENS)

# offset_mapping gives each token's (start, end) character position in the contract.
enc = tokenizer([c["text"] for c in contracts], add_special_tokens=False,
                return_offsets_mapping=True)
for c, offsets in zip(contracts, enc["offset_mapping"]):
    c["offsets"] = offsets
    c["n_tokens"] = len(offsets)

n_tokens = pd.Series([c["n_tokens"] for c in contracts])
n_chars = sum(len(c["text"]) for c in contracts)

print("characters per token:", round(n_chars / n_tokens.sum(), 2))
print()
print(n_tokens.describe().round(0).to_string())
print()
print("contracts longer than", MAX_TOKENS, "tokens:", (n_tokens > MAX_TOKENS).sum(), "of", len(contracts))

[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


tokens of contract text per chunk: 510
characters per token: 4.09

count       408.0
mean      13208.0
std       14855.0
min         236.0
25%        3963.0
50%        8482.0
75%       16255.0
max      121051.0

contracts longer than 510 tokens: 401 of 408


In [3]:
# 2. The chunking strategies
def token_windows(offsets, size, overlap):
    """Character (start, end) of each window of `size` tokens."""
    if not offsets:
        return []
    step = size - overlap
    chunks = []
    for i in range(0, len(offsets), step):
        window = offsets[i:i + size]
        chunks.append((window[0][0], window[-1][1]))
        if i + size >= len(offsets):
            break
    return chunks


def paragraphs(text):
    """Character (start, end) of each block of text separated by a blank line."""
    blocks, start = [], 0
    for m in re.finditer(r"\n\s*\n", text):
        if text[start:m.start()].strip():
            blocks.append((start, m.start()))
        start = m.end()
    if text[start:].strip():
        blocks.append((start, len(text)))
    return blocks


def paragraph_chunks(text, offsets, size):
    """Pack whole paragraphs into chunks of at most `size` tokens.

    A single paragraph longer than `size` falls back to plain token windows.
    Returns the chunks and how many paragraphs needed that fallback.
    """
    token_starts = [s for s, _ in offsets]
    chunks, current, n_split = [], None, 0       # current = [start, end, n_tokens]
    for p_start, p_end in paragraphs(text):
        n = bisect_left(token_starts, p_end) - bisect_left(token_starts, p_start)
        if n > size:
            if current:
                chunks.append(tuple(current[:2]))
                current = None
            i = bisect_left(token_starts, p_start)
            chunks += token_windows(offsets[i:i + n], size, overlap=0)
            n_split += 1
            continue
        if current and current[2] + n > size:
            chunks.append(tuple(current[:2]))
            current = None
        if current is None:
            current = [p_start, p_end, n]
        else:
            current[1], current[2] = p_end, current[2] + n
    if current:
        chunks.append(tuple(current[:2]))
    return chunks, n_split


def intact(chunks, spans):
    """How many spans fit entirely inside at least one chunk."""
    return sum(any(cs <= s and e <= ce for cs, ce in chunks) for s, e in spans)

In [4]:
# 3. Compare: how many chunks, and how many clauses survive in one piece?
rows = []

for overlap in [0, 64, 128, 256]:
    n_chunks = n_intact = 0
    for c in contracts:
        chunks = token_windows(c["offsets"], MAX_TOKENS, overlap)
        n_chunks += len(chunks)
        n_intact += intact(chunks, c["spans"])
    rows.append({"strategy": f"token windows, overlap {overlap}",
                 "chunks": n_chunks,
                 "spans_intact_%": round(100 * n_intact / total_spans, 1)})

n_chunks = n_intact = n_split = n_paragraphs = 0
for c in contracts:
    chunks, split = paragraph_chunks(c["text"], c["offsets"], MAX_TOKENS)
    n_chunks += len(chunks)
    n_intact += intact(chunks, c["spans"])
    n_split += split
    n_paragraphs += len(paragraphs(c["text"]))
rows.append({"strategy": "paragraph-aware",
             "chunks": n_chunks,
             "spans_intact_%": round(100 * n_intact / total_spans, 1)})

comparison = pd.DataFrame(rows)
print(comparison.to_string(index=False))
print()
print(f"paragraphs too long for one chunk: {n_split} of {n_paragraphs} "
      f"({100 * n_split / n_paragraphs:.1f}%)")

                  strategy  chunks  spans_intact_%
  token windows, overlap 0   10777            88.7
 token windows, overlap 64   12219            95.1
token windows, overlap 128   14181            97.8
token windows, overlap 256   21006            99.2
           paragraph-aware   14045            96.3

paragraphs too long for one chunk: 1868 of 62233 (3.0%)


In [5]:
# Some spans are longer than one chunk, so no strategy can keep them whole.
too_long = 0
for c in contracts:
    token_starts = [s for s, _ in c["offsets"]]
    for s, e in c["spans"]:
        if bisect_left(token_starts, e) - bisect_left(token_starts, s) > MAX_TOKENS:
            too_long += 1

print(f"spans longer than {MAX_TOKENS} tokens: {too_long} ({100 * too_long / total_spans:.2f}%)")
print(f"best possible spans_intact_%: {100 * (total_spans - too_long) / total_spans:.2f}")

spans longer than 510 tokens: 26 (0.23%)
best possible spans_intact_%: 99.77


**Reading the table**

- **Overlap matters more than anything else.** Going from 0 to 128 tokens of overlap
  moves clauses kept whole from 88.7% to 97.8%.
- **Paragraph-aware loses to plain windows at the same cost.** It makes about as many
  chunks as 128-overlap windows (14,045 vs 14,181) but keeps fewer clauses whole
  (96.3% vs 97.8%). Blank lines are not reliable clause boundaries in this text,
  and 3% of "paragraphs" are too long to fit anyway.
- **256 overlap is past the point of diminishing returns.** It gains 1.4 points over 128
  but makes 48% more chunks, which means 48% more training and inference time.

In [6]:
# 4. The chosen strategy on the shortest and longest contracts
OVERLAP = 128      # chosen from the table above

by_length = sorted(contracts, key=lambda c: c["n_tokens"])

for label, c in [("SHORTEST", by_length[0]), ("LONGEST", by_length[-1])]:
    chunks = token_windows(c["offsets"], MAX_TOKENS, OVERLAP)
    print(label)
    print("  ", c["title"][:70])
    print("   words: ", len(c["text"].split()))
    print("   tokens:", c["n_tokens"])
    print("   chunks:", len(chunks))
    print("   spans intact:", intact(chunks, c["spans"]), "of", len(c["spans"]))
    print()

SHORTEST
   NELNETINC_04_08_2020-EX-1-JOINT FILING AGREEMENT
   words:  170
   tokens: 236
   chunks: 1
   spans intact: 4 of 4

LONGEST
   MANUFACTURERSSERVICESLTD_06_05_2000-EX-10.14-OUTSOURCING AGREEMENT
   words:  41962
   tokens: 121051
   chunks: 317
   spans intact: 65 of 65



In [7]:
# 5. A clause that no-overlap chunking cuts in half, and overlap keeps whole
example = None
for c in contracts:
    no_overlap = token_windows(c["offsets"], MAX_TOKENS, 0)
    with_overlap = token_windows(c["offsets"], MAX_TOKENS, OVERLAP)
    for s, e in c["spans"]:
        if not intact(no_overlap, [(s, e)]) and intact(with_overlap, [(s, e)]):
            example = (c, s, e, no_overlap)
            break
    if example:
        break

c, s, e, no_overlap = example
cut = next(ce for cs, ce in no_overlap if s < ce < e)   # where the chunk boundary falls

def clean(t):
    return " ".join(t.split())

print(c["title"][:70])
print(f"clause at characters {s}-{e}; no-overlap chunk boundary at {cut}")
print()
print("END OF CHUNK 1:  ...", clean(c["text"][s:cut])[-200:])
print()
print("START OF CHUNK 2:", clean(c["text"][cut:e])[:200], "...")

LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT
clause at characters 31058-31270; no-overlap chunk boundary at 31207

END OF CHUNK 1:  ... Unless earlier terminated otherwise provided therein, this Agreement, subject to the commencement date

START OF CHUNK 2: established in Section 1.3, shall be effective immediately. ...


## Decision

| | |
|---|---|
| **Strategy** | Sliding token windows |
| **Chunk size** | 510 tokens of contract text (512 minus `<s>` and `</s>`) |
| **Overlap** | 128 tokens (each window starts 382 tokens after the previous one) |
| **Tokenizer** | `distilroberta-base` |
| **Boundary policy** | Cut on token boundaries, ignoring paragraphs. The last window ends at the end of the contract and may be shorter. |
| **Result on train** | 14,181 chunks; 97.8% of annotated clauses fall entirely inside at least one chunk |

**Why**

- 401 of 408 train contracts are longer than one window, so chunking is required.
- 128 overlap keeps 97.8% of clauses whole, against 88.7% with no overlap.
- Paragraph-aware chunking costs the same and does worse (96.3%).
- 256 overlap would add 48% more chunks for 1.4 more points.

**Known limitations**

- **26 clauses (0.23%) are longer than 510 tokens** and cannot fit in any chunk. The
  best any strategy could do is 99.77%.
- **Windows cut mid-sentence.** That is fine for detecting whether a clause is present,
  but the evidence text shown to a reviewer later should be the full annotated span,
  not the raw chunk.
- **Overlap duplicates text.** A clause inside an overlap appears in two chunks, so
  chunk-level predictions must be merged back to contract level before scoring, or the
  same clause is counted twice. This belongs to #24 and the evaluation code.
- **Page counts are not in the data.** The issue asks about 100+ page contracts; the
  longest train contract by tokens has 41,962 words and 121,051 tokens, splits into
  317 chunks, and keeps all 65 of its clauses whole.
- **Train split only.** These numbers were not checked on test, deliberately.

**For #23:** implement `token_windows` above as `chunk_contract(text)` in `src/ingest/`,
with `MAX_TOKENS = 510` and `OVERLAP = 128`. The character offsets it returns are what
#24 needs to map spans onto chunks.